> **Current notebook role — 2026-11-12.** This notebook retains the upstream stage code used by the manuscript implementation. Selected figure production uses `run.py` in `code_release/iCCA-molecular-subtyping` or the public source repository. See `docs/NOTEBOOK_STATUS.md` in that code repository for stage ownership. Earlier execution outputs are preserved in the archived notebook snapshot.

# Step 2 — Comprehensive transcriptomic characterization

## tl;dr

This notebook owns corrected DEG contrasts, ranked pathway analysis, TF activity, TME, metabolic and differentiation programs, TIDE imports, and intermediate-state transcriptomic characterization.

Historical bulk-supplementary material was consolidated here. The former notebook and outputs are retained only under `archive/pipeline_migration_20260715/`; they are not active dependencies. Clinical and survival analyses are intentionally downstream and are owned by `6-clinical-survival.ipynb`.


## Context & Methods

Principal-state statistics use only C1/C2/C3 (n=497), include Study adjustment where applicable, and define positive effects as subtype minus others. Intermediate states are retained for supplementary continuity analyses. Biological suffixes are interpretations, not subtype identifiers; C2 remains without a suffix.

### Key assumptions

- The Step1 clustering gate has passed and subtype assignments are locked inputs.
- Historical reversed DEG and legacy TF/EPIC files are excluded.
- TIDE, mutation counts, and treatment-response projections are hypothesis-generating.
- Clinical and survival analyses are supportive imports until their own downstream gates pass.

In [ ]:
suppressPackageStartupMessages({
  library(data.table)
  library(ggplot2)
  library(openxlsx)
  library(survival)
  library(survminer)
  library(ggalluvial)
  library(ComplexHeatmap)
  library(circlize)
})

project_root <- normalizePath(getwd(), winslash = "/", mustWork = TRUE)
out_dir <- file.path(project_root, "2-TranscriptomicCharacterization")
data_dir <- file.path(out_dir, "data")
fig_dir <- file.path(out_dir, "figures")
report_dir <- file.path(out_dir, "reports")
dir.create(data_dir, recursive = TRUE, showWarnings = FALSE)
dir.create(fig_dir, recursive = TRUE, showWarnings = FALSE)
dir.create(report_dir, recursive = TRUE, showWarnings = FALSE)

main_subtypes <- c("C1", "C2", "C3")
intermediate_subtypes <- c("C1_C2", "C1_C3", "C2_C3")
subtype_levels <- c(main_subtypes, intermediate_subtypes)
palette_contract <- fread(file.path(project_root, "docs", "subtype_color_dictionary.csv"))
stopifnot(identical(palette_contract$FinalSubtype, subtype_levels))
subtype_key <- palette_contract[, .(
  FinalSubtype, DisplayLabel, SubtypeClass, Color = ColorHex
)]

theme_pub <- function(base_size = 11) {
  theme_classic(base_size = base_size) +
    theme(
      strip.background = element_rect(fill = "grey95", color = "grey70"),
      axis.text.x = element_text(angle = 45, hjust = 1),
      legend.position = "right"
    )
}

subtype_colors <- setNames(subtype_key$Color, subtype_key$FinalSubtype)
display_colors <- setNames(subtype_key$Color, subtype_key$DisplayLabel)

num <- function(x) suppressWarnings(as.numeric(as.character(x)))

load_r_object <- function(path, object_name = NULL) {
  e <- new.env()
  loaded <- load(path, envir = e)
  if (!is.null(object_name)) return(e[[object_name]])
  e[[loaded[1]]]
}

zscore_rows <- function(mat) {
  z <- t(scale(t(mat)))
  z[is.na(z)] <- 0
  z
}

score_gene_sets <- function(expr, gene_sets, method = c("zmean", "ssgsea")) {
  method <- match.arg(method)
  gene_sets <- lapply(gene_sets, function(g) intersect(unique(g), rownames(expr)))
  gene_sets <- gene_sets[lengths(gene_sets) >= 3]
  if (method == "zmean") {
    z <- zscore_rows(expr)
    res <- sapply(gene_sets, function(g) colMeans(z[g, , drop = FALSE], na.rm = TRUE))
    return(as.data.table(t(res), keep.rownames = "Signature"))
  }
  if (!requireNamespace("GSVA", quietly = TRUE)) stop("GSVA is not installed")
  gsva_res <- tryCatch({
    GSVA::gsva(expr, gene_sets, method = "ssgsea", verbose = FALSE)
  }, error = function(e) {
    if (exists("ssgseaParam", where = asNamespace("GSVA"), inherits = FALSE)) {
      param <- GSVA::ssgseaParam(expr, gene_sets)
      GSVA::gsva(param, verbose = FALSE)
    } else {
      stop(e)
    }
  })
  as.data.table(gsva_res, keep.rownames = "Signature")
}

to_long_scores <- function(score_dt, sample_cols = NULL) {
  if (is.null(sample_cols)) sample_cols <- setdiff(names(score_dt), "Signature")
  melt(score_dt, id.vars = "Signature", measure.vars = sample_cols,
       variable.name = "Sample", value.name = "Score")
}

compare_by_subtype <- function(score_long, meta, group_col = "FinalSubtype") {
  dt <- merge(score_long, meta[, .(Sample, FinalSubtype, SubtypeClass, MainSubtypeForPlot)], by = "Sample")
  summaries <- dt[, .(
    N = .N,
    Mean = mean(Score, na.rm = TRUE),
    Median = median(Score, na.rm = TRUE),
    SD = sd(Score, na.rm = TRUE)
  ), by = .(Signature, Group = get(group_col))]
  kw <- dt[, {
    x <- .SD[!is.na(Score) & !is.na(get(group_col))]
    if (length(unique(x[[group_col]])) >= 2) {
      p <- kruskal.test(Score ~ get(group_col), data = x)$p.value
    } else p <- NA_real_
    .(KruskalP = p)
  }, by = Signature]
  kw[, FDR := p.adjust(KruskalP, method = "BH")]
  list(scores = dt, summary = summaries, kruskal = kw)
}

write_table_set <- function(prefix, result) {
  fwrite(result$scores, file.path(data_dir, paste0(prefix, "_scores_long.csv")))
  fwrite(result$summary, file.path(data_dir, paste0(prefix, "_summary.csv")))
  fwrite(result$kruskal, file.path(data_dir, paste0(prefix, "_kruskal.csv")))
}

plot_score_boxplot <- function(dt, x_col, filename, title) {
  plot_dt <- copy(as.data.table(dt))
  clean_labels <- gsub("^HALLMARK_", "", as.character(plot_dt$Signature))
  clean_labels <- gsub("_", " ", clean_labels)
  plot_dt[, SignatureLabel := vapply(clean_labels, function(x) {
    paste(strwrap(x, width = 24), collapse = "\n")
  }, character(1))]
  panel_rows <- ceiling(uniqueN(plot_dt$SignatureLabel) / 3)
  p <- ggplot(plot_dt, aes(x = .data[[x_col]], y = Score, fill = .data[[x_col]])) +
    geom_boxplot(outlier.size = 0.5, alpha = 0.85) +
    facet_wrap(~ SignatureLabel, scales = "free_y", ncol = 3) +
    scale_fill_manual(values = subtype_colors, na.value = "grey70") +
    theme_pub(10) +
    theme(legend.position = "none", strip.text = element_text(size = 8.5)) +
    labs(title = title, x = NULL, y = "Score")
  ggsave(file.path(fig_dir, filename), p, width = 9, height = max(5.5, 2.15 * panel_rows), dpi = 300)
  p
}


step1_table_dir <- file.path(project_root, "1-Clustering", "publication", "tables")
step1_figure_dir <- file.path(project_root, "1-Clustering", "publication", "figures")
refresh_core_transcriptomics <- identical(Sys.getenv("SCLUCID_TRANSCRIPTOMIC_REFRESH", "0"), "1")
cat("Execution mode:", ifelse(refresh_core_transcriptomics, "core transcriptomic refresh", "validation plus derived scoring"), "\n")

## Data

In [ ]:
master_file <- file.path(project_root, "governance/data/subtype_master_table.csv")
stopifnot(file.exists(master_file))
subtype_master <- fread(master_file)
subtype_master <- merge(subtype_master, subtype_key[, .(FinalSubtype, DisplayLabel, Color)],
                        by = "FinalSubtype", all.x = TRUE, sort = FALSE)
subtype_master[, FinalSubtype := factor(FinalSubtype, levels = subtype_levels)]
subtype_master[, DisplayLabel := factor(DisplayLabel, levels = subtype_key$DisplayLabel)]
fwrite(subtype_key, file.path(data_dir, "subtype_label_color_dictionary.csv"))

expr <- load_r_object(file.path(project_root, "1-Clustering/data/combined_expr_after_correction.rds"),
                      "corrected_expr_limma")
expr <- as.matrix(expr)
raw_expr <- load_r_object(file.path(project_root, "1-Clustering/data/combined_expr_before_correction.rds"),
                          "combined_expr")
raw_expr <- as.matrix(raw_expr)

common_samples <- intersect(subtype_master$Sample, colnames(expr))
expr <- expr[, common_samples, drop = FALSE]
raw_expr <- raw_expr[, common_samples, drop = FALSE]
subtype_master <- subtype_master[match(common_samples, Sample)]

main_meta <- subtype_master[SubtypeClass == "Main"]
all_meta <- subtype_master[FinalSubtype %in% c(main_subtypes, intermediate_subtypes)]

cat("Expression matrix:", nrow(expr), "genes x", ncol(expr), "samples\n")
print(subtype_master[, .N, by = .(FinalSubtype, SubtypeClass)][order(SubtypeClass, FinalSubtype)])

## Results

### 1. Corrected principal-state differential expression

In [ ]:
if (refresh_core_transcriptomics) {
  scripts <- file.path(project_root, "1-Clustering", "scripts", c(
    "run_step1_publication_repair.R",
    "run_step1_tf_activity.R",
    "run_step1_tme_concordance.R",
    "finalize_step1_audit.R",
    "run_step1_source_figures.R"
  ))
  stopifnot(all(file.exists(scripts)))
  for (script in scripts) {
    cat("Running", basename(script), "\n")
    sys.source(script, envir = new.env(parent = globalenv()))
  }
}

deg <- fread(file.path(step1_table_dir, "step1_main_only_study_adjusted_deg.csv"))
deg_summary <- deg[, .(
  Higher = sum(FDR < 0.05 & logFC_subtype_minus_others >= 1),
  Lower = sum(FDR < 0.05 & logFC_subtype_minus_others <= -1)
), by = Subtype]
stopifnot(nrow(deg) == 31593L, all(deg$PositiveDirection == paste0("Higher in ", deg$Subtype)))
deg_summary

### 2. Ranked pathways and thresholded enrichment

In [ ]:
gsea <- fread(file.path(step1_table_dir, "step1_ranked_gsea_main_only_study_adjusted.csv"))
ora <- fread(file.path(step1_table_dir, "step1_thresholded_ora_main_only_study_adjusted.csv"))
gsea_top <- gsea[padj < 0.05][order(Subtype, Collection, padj, -abs(NES)), head(.SD, 5L), by = .(Subtype, Collection)]
fwrite(gsea_top, file.path(data_dir, "ranked_gsea_top5_by_subtype_collection.csv"))
gsea_top[, .(Subtype, Collection, pathway, NES, padj)]

### 3. Study-adjusted DoRothEA TF activity

In [ ]:
tf_long <- fread(file.path(step1_table_dir, "step1_dorothea_ulm_activity_long.csv"))
tf_diff <- fread(file.path(step1_table_dir, "step1_dorothea_tf_activity_main_only_study_adjusted.csv"))
tf_top <- tf_diff[FDR < 0.05][order(Subtype, -abs(ActivityDifferenceSubtypeMinusOthers)), head(.SD, 12L), by = Subtype]
fwrite(tf_top, file.path(data_dir, "dorothea_top_tf_by_subtype.csv"))

tf_plot_data <- merge(
  tf_long[TF %in% tf_top$TF],
  subtype_master[, .(Sample, FinalSubtype, SubtypeClass)],
  by = "Sample"
)[SubtypeClass == "Main"]
p_tf <- ggplot(tf_plot_data, aes(FinalSubtype, Activity, fill = FinalSubtype)) +
  geom_boxplot(outlier.size = 0.35, alpha = 0.85) +
  facet_wrap(~TF, scales = "free_y") +
  scale_fill_manual(values = subtype_colors[main_subtypes]) +
  theme_pub(8) + theme(legend.position = "none") +
  labs(title = "Study-adjusted DoRothEA regulators", x = NULL, y = "ULM activity")
ggsave(file.path(fig_dir, "dorothea_top_tf_main_subtypes.png"), p_tf, width = 12, height = 8, dpi = 300)
tf_top[, .(TF, Subtype, ActivityDifferenceSubtypeMinusOthers, FDR)]

### 4. TME and immune-program concordance

In [ ]:
tme_long <- fread(file.path(step1_table_dir, "step1_tme_concordance_scores_long.csv"))
tme_tests <- fread(file.path(step1_table_dir, "step1_tme_c2_vs_others_tests.csv"))
tme_status <- fread(file.path(step1_table_dir, "step1_tme_method_status.csv"))
fwrite(tme_tests, file.path(data_dir, "tme_c2_vs_others_tests.csv"))
tme_status
tme_tests[order(FDR), .(Method, Score, C2Median, OthersMedian, RankBiserialC2MinusOthers, FDR)]

### 5. Classifier-signature membership context

In [ ]:
signature_panels <- readRDS(file.path(project_root, "5-ClassifierValidation/selected_signature_genes.rds"))
required_panels <- c("operational_123", "nested_31", "compact_15", "historical_interpretive_31")
stopifnot(all(required_panels %in% names(signature_panels)))

signature_sources <- rbindlist(lapply(required_panels, function(panel) {
  data.table(Gene = signature_panels[[panel]], Panel = panel)
}))
signature_table <- signature_sources[, .(
  InOperational123 = "operational_123" %in% Panel,
  InNested31 = "nested_31" %in% Panel,
  InCompact15 = "compact_15" %in% Panel,
  InHistoricalInterpretive31 = "historical_interpretive_31" %in% Panel,
  NumPanels = uniqueN(Panel)
), by = Gene][order(-NumPanels, Gene)]
signature_table[, PresentInExpression := Gene %in% rownames(expr)]

# Preserve the historical output path because it is listed in the Step2
# manifest, but the columns now explicitly describe the current panel contract.
fwrite(signature_table, file.path(data_dir, "signature_31_gene_membership_table.csv"))
write.xlsx(signature_table, file.path(data_dir, "signature_31_gene_membership_table.xlsx"), overwrite = TRUE)
signature_table[InNested31 == TRUE]

### 6. Metabolic pathway programs

In [ ]:
manual_metabolic_sets <- list(
  Glycolysis = c("HK1","HK2","GPI","PFKL","PFKM","PFKP","ALDOA","ALDOB","ALDOC","GAPDH","PGK1","PGAM1","ENO1","ENO2","PKM","LDHA","LDHB","SLC2A1","SLC2A3","PDK1"),
  OXPHOS = c("NDUFA1","NDUFA2","NDUFB3","NDUFS1","NDUFS2","SDHA","SDHB","UQCRC1","UQCRC2","COX4I1","COX5A","COX6A1","ATP5F1A","ATP5F1B","ATP5MC1","ATP5MC2"),
  LipidMetabolism = c("ACACA","ACACB","FASN","SCD","CPT1A","CPT2","ACOX1","ACSL1","ACSL3","ACSL5","PPARA","PPARG","APOA1","APOA2","FABP1","FABP5"),
  BileAcidMetabolism = c("CYP7A1","CYP8B1","CYP27A1","CYP3A4","CYP3A5","BAAT","SLC10A1","SLC10A2","ABCB11","ABCC2","NR1H4","NR1I2"),
  XenobioticMetabolism = c("CYP1A1","CYP1A2","CYP2A6","CYP2C9","CYP2D6","CYP2E1","CYP3A4","CYP3A5","UGT1A1","UGT2B7","GSTA1","GSTM1","GSTP1","NQO1","ALDH1A1")
)

msig_sets <- list()
if (requireNamespace("msigdbr", quietly = TRUE)) {
  msig <- tryCatch(msigdbr::msigdbr(species = "Homo sapiens", collection = "H"), error = function(e) NULL)
  if (!is.null(msig)) {
    wanted <- c("HALLMARK_GLYCOLYSIS", "HALLMARK_OXIDATIVE_PHOSPHORYLATION",
                "HALLMARK_FATTY_ACID_METABOLISM", "HALLMARK_BILE_ACID_METABOLISM",
                "HALLMARK_XENOBIOTIC_METABOLISM")
    msig_sets <- split(msig$gene_symbol[msig$gs_name %in% wanted], msig$gs_name[msig$gs_name %in% wanted])
  }
}

metabolic_sets <- c(manual_metabolic_sets, msig_sets)
metabolic_z <- score_gene_sets(expr, metabolic_sets, method = "zmean")
metabolic_long <- to_long_scores(metabolic_z)

metabolic_main <- compare_by_subtype(metabolic_long, main_meta, "FinalSubtype")
metabolic_all <- compare_by_subtype(metabolic_long, all_meta, "FinalSubtype")
write_table_set("metabolic_main_subtypes", metabolic_main)
write_table_set("metabolic_all_subtypes", metabolic_all)

plot_score_boxplot(metabolic_main$scores, "FinalSubtype",
                   "metabolic_scores_main_subtypes_boxplot.png",
                   "Metabolic pathway scores in main subtypes")
plot_score_boxplot(metabolic_all$scores, "FinalSubtype",
                   "metabolic_scores_all_subtypes_boxplot.png",
                   "Metabolic pathway scores including intermediate states")

### 7. Stemness, differentiation, and proliferation programs

In [ ]:
stemness_sets <- list(
  StemnessCore = c("SOX2","POU5F1","NANOG","KLF4","MYC","PROM1","ALDH1A1","ALDH1A3","CD44","CD24","LGR5","EPCAM","KRT19"),
  EMT_pEMT = c("EPCAM","KRT8","KRT18","KRT19","VIM","FN1","SNAI1","SNAI2","ZEB1","ZEB2","TWIST1","ITGA5","CDH2"),
  ProliferationDediff = c("MKI67","TOP2A","PCNA","BIRC5","UBE2C","MCM2","MCM5","CCNB1","E2F1","FOXM1"),
  HepaticDifferentiation = c("ALB","TTR","APOA1","APOA2","CYP3A4","CYP2E1","HNF4A","CPS1","ARG1")
)

stemness_z <- score_gene_sets(expr, stemness_sets, method = "zmean")
stemness_long <- to_long_scores(stemness_z)

# A lightweight CytoTRACE-like bulk proxy: number of genes above the cohort median after batch correction.
# This is not a replacement for single-cell CytoTRACE; interpret as transcriptional diversity / dedifferentiation-like signal.
gene_medians <- apply(expr, 1, median, na.rm = TRUE)
expressed_counts <- colSums(expr > gene_medians, na.rm = TRUE)
expressed_gene_count <- as.data.table(as.list(expressed_counts))
expressed_gene_count[, Signature := "CytoTRACE_like_expressed_gene_count"]
setcolorder(expressed_gene_count, c("Signature", colnames(expr)))
stemness_long <- rbind(stemness_long, to_long_scores(expressed_gene_count), fill = TRUE)


stemness_main <- compare_by_subtype(stemness_long, main_meta, "FinalSubtype")
stemness_all <- compare_by_subtype(stemness_long, all_meta, "FinalSubtype")
write_table_set("stemness_main_subtypes", stemness_main)
write_table_set("stemness_all_subtypes", stemness_all)

plot_score_boxplot(stemness_main$scores, "FinalSubtype",
                   "stemness_scores_main_subtypes_boxplot.png",
                   "Stemness / dedifferentiation-like scores in main subtypes")
plot_score_boxplot(stemness_all$scores, "FinalSubtype",
                   "stemness_scores_all_subtypes_boxplot.png",
                   "Stemness / dedifferentiation-like scores including intermediate states")

### 8. Study-aware TIDE refresh (hypothesis-generating)

The active TIDE analysis is generated by `2-TranscriptomicCharacterization/scripts/run_step2_tide_refresh.py`. It uses all 497 principal-state samples, centers each gene within Study before calling TIDEpy 1.3.9 (`cancer=Other`, `--ignore_norm`), and then fits Study-adjusted subtype models plus study-specific random-effects summaries. Set `SCLUCID_TIDE_REFRESH=1` to rerun it.

TIDE is not calibrated or clinically validated for iCCA. “Responder” is therefore reported only as an algorithm-derived exploratory label; it is not an observed immunotherapy outcome and cannot justify the term “immune-sensitive.”

In [ ]:
tide_refresh <- identical(Sys.getenv("SCLUCID_TIDE_REFRESH", "0"), "1")
tide_python <- "/Users/luye/micromamba/envs/scrna-env/bin/python"
tide_script <- file.path(project_root, "2-TranscriptomicCharacterization/scripts/run_step2_tide_refresh.py")
if (tide_refresh) {
  tide_env <- c("MPLCONFIGDIR=/tmp/sclucid-mpl", "NUMBA_CACHE_DIR=/tmp/sclucid-numba")
  status <- system2("env", c(tide_env, tide_python, tide_script), stdout = TRUE, stderr = TRUE)
  cat(paste(status, collapse = "\n"), "\n")
  if (!is.null(attr(status, "status")) && attr(status, "status") != 0) stop("Step2 TIDE refresh failed")
}
tide_table_dir <- file.path(project_root, "2-TranscriptomicCharacterization/publication/tables")
tide_gate_file <- file.path(tide_table_dir, "step2_tide_release_gate_status.csv")
stopifnot(file.exists(tide_gate_file))
tide_gate <- fread(tide_gate_file)
stopifnot(all(tide_gate$Pass))
tide_summary <- fread(file.path(tide_table_dir, "step2_tide_subtype_summary.csv"))
tide_adjusted <- fread(file.path(tide_table_dir, "step2_tide_adjusted_pairwise.csv"))
tide_meta <- fread(file.path(tide_table_dir, "step2_tide_meta_analysis.csv"))
list(
  release_gate = tide_gate,
  descriptive_summary = tide_summary,
  adjusted_TIDE = tide_adjusted[Metric == "TIDE"],
  study_meta_TIDE = tide_meta[Metric == "TIDE"]
)

### 9. Intermediate-state continuity and supporting clinical/mutation summaries

In [ ]:
clinical <- copy(subtype_master)
clinical[, time_days := num(`Overall survival (days)`)]
clinical[, status := num(`Vital state (1=Dead)`)]
clinical[, Age_num2 := num(Age_num)]
clinical[, TumorSize_num := num(`Tumor size (cm)`)]

surv_dt <- clinical[!is.na(time_days) & !is.na(status) & time_days > 0]
surv_dt[, AnalysisGroup := factor(as.character(FinalSubtype), levels = subtype_levels)]
fit_all <- survfit(Surv(time_days, status) ~ AnalysisGroup, data = surv_dt)
surv_palette <- unname(subtype_colors[subtype_levels[subtype_levels %in% as.character(unique(surv_dt$AnalysisGroup))]])
km_plot <- ggsurvplot(
  fit_all, data = surv_dt, pval = TRUE, risk.table = TRUE,
  palette = surv_palette,
  title = "Survival across main and intermediate states"
)
ggsave(file.path(fig_dir, "survival_main_and_intermediate_states.png"), km_plot$plot, width = 7, height = 5, dpi = 300)
ggsave(file.path(fig_dir, "survival_main_and_intermediate_states_risktable.png"), km_plot$table, width = 7, height = 3, dpi = 300)

cox_intermediate <- coxph(Surv(time_days, status) ~ relevel(factor(SubtypeClass), ref = "Main"), data = surv_dt)
cox_intermediate_summary <- data.table(
  Term = rownames(summary(cox_intermediate)$coefficients),
  HR = summary(cox_intermediate)$conf.int[, "exp(coef)"],
  Lower95 = summary(cox_intermediate)$conf.int[, "lower .95"],
  Upper95 = summary(cox_intermediate)$conf.int[, "upper .95"],
  pvalue = summary(cox_intermediate)$coefficients[, "Pr(>|z|)"]
)
fwrite(cox_intermediate_summary, file.path(data_dir, "intermediate_vs_main_survival_cox.csv"))

maf_file <- file.path(project_root, "0-Data/combined_maf_data.csv")
if (file.exists(maf_file)) {
  legacy_tmb_file <- file.path(data_dir, "tmb_by_final_subtype.csv")
  if (file.exists(legacy_tmb_file)) {
    file.copy(legacy_tmb_file, file.path(report_dir, "deprecated_tmb_by_final_subtype_previous.csv"),
              overwrite = TRUE)
    unlink(legacy_tmb_file)
  }
  maf <- fread(maf_file)
  nonsyn <- maf[!Variant_Classification %in% c("Silent", "Intron", "IGR", "RNA", "3'UTR", "5'UTR")]
  mutation_counts <- nonsyn[, .(NonsynonymousMutationCount = .N), by = .(Sample = Tumor_Sample_Barcode)]
  mutation_counts <- merge(mutation_counts, subtype_master[, .(Sample, FinalSubtype, DisplayLabel, SubtypeClass)], by = "Sample")
  mutation_coverage <- subtype_master[, .(SubtypeN = .N), by = .(FinalSubtype, DisplayLabel, SubtypeClass)]
  mutation_coverage <- merge(mutation_coverage,
                             mutation_counts[, .(MutationDataN = .N), by = .(FinalSubtype, DisplayLabel, SubtypeClass)],
                             by = c("FinalSubtype", "DisplayLabel", "SubtypeClass"), all.x = TRUE)
  mutation_coverage[is.na(MutationDataN), MutationDataN := 0L]
  mutation_coverage[, CoverageFraction := MutationDataN / SubtypeN]
  fwrite(mutation_counts, file.path(data_dir, "nonsynonymous_mutation_count_by_final_subtype.csv"))
  fwrite(mutation_coverage, file.path(data_dir, "mutation_data_coverage_by_final_subtype.csv"))
  p_tmb <- ggplot(mutation_counts, aes(FinalSubtype, NonsynonymousMutationCount, fill = FinalSubtype)) +
    geom_boxplot(outlier.size = 0.5, alpha = 0.85) +
    scale_fill_manual(values = subtype_colors) +
    theme_pub(10) +
    theme(legend.position = "none") +
    labs(title = "Non-synonymous mutation count in matched samples", x = NULL,
         y = "Non-synonymous mutation count",
         caption = "Exploratory count; not normalized tumor mutational burden.")
  ggsave(file.path(fig_dir, "nonsynonymous_mutation_count_by_final_subtype.png"), p_tmb, width = 7, height = 4.8, dpi = 300)
}

clinical_vars <- c("Age_num2", "TumorSize_num", "Stage", "Sex", "iCCA classification", "lasso_pred",
                   "HBV infection (1=Positive)", "Vascular invasion (1=Positive)")
continuous_vars <- c("Age_num2", "TumorSize_num")
clinical_summary <- rbindlist(lapply(clinical_vars, function(v) {
  if (!v %in% names(clinical)) return(NULL)
  if (v %in% continuous_vars) {
    ans <- clinical[, .(N = sum(!is.na(get(v))),
                        Mean = as.numeric(mean(get(v), na.rm = TRUE)),
                        Median = as.numeric(median(get(v), na.rm = TRUE))),
                    by = .(FinalSubtype, SubtypeClass)]
    ans[, Variable := v]
    setcolorder(ans, c("FinalSubtype", "SubtypeClass", "Variable", setdiff(names(ans), c("FinalSubtype", "SubtypeClass", "Variable"))))
    ans
  } else {
    tmp <- clinical[, .(FinalSubtype, SubtypeClass, Level = as.character(get(v)))]
    ans <- tmp[, .N, by = .(FinalSubtype, SubtypeClass, Level)]
    ans[, Variable := v]
    setcolorder(ans, c("FinalSubtype", "SubtypeClass", "Variable", "Level", "N"))
    ans
  }
}), fill = TRUE)
fwrite(clinical_summary, file.path(data_dir, "clinical_features_by_final_subtype.csv"))

set.seed(123)
clinical_tests <- rbindlist(lapply(clinical_vars, function(v) {
  if (!v %in% names(clinical)) return(NULL)
  x <- clinical[SubtypeClass == "Main" & !is.na(get(v))]
  if (nrow(x) == 0 || uniqueN(x$FinalSubtype) < 2) return(NULL)
  if (v %in% continuous_vars) {
    p <- kruskal.test(x[[v]] ~ x$FinalSubtype)$p.value
    return(data.table(Variable = v, AnalysisSet = "Main", Test = "Kruskal-Wallis", PValue = p))
  }
  tab <- table(x[[v]], x$FinalSubtype)
  if (nrow(tab) < 2 || ncol(tab) < 2) return(NULL)
  expected <- suppressWarnings(chisq.test(tab)$expected)
  if (any(expected < 5)) {
    p <- fisher.test(tab, simulate.p.value = TRUE, B = 10000)$p.value
    method <- "Fisher exact (simulated)"
  } else {
    p <- chisq.test(tab)$p.value
    method <- "Chi-square"
  }
  data.table(Variable = v, AnalysisSet = "Main", Test = method, PValue = p)
}), fill = TRUE)
clinical_tests[, FDR := p.adjust(PValue, method = "BH")]
fwrite(clinical_tests, file.path(data_dir, "clinical_association_tests_main_subtypes.csv"))

### 10. Relationship to published subtype and duct annotations

In [ ]:
sankey_dt <- copy(subtype_master)
sankey_dt[, SiaSubclass := fifelse(!is.na(Subclass) & Subclass != "", Subclass, "Unknown")]
sankey_dt[, IGSubtype := fifelse(!is.na(`Subclass-add`) & `Subclass-add` != "", `Subclass-add`, "Unknown")]
sankey_dt[, DuctType := fifelse(!is.na(lasso_pred) & lasso_pred != "", lasso_pred,
                                fifelse(!is.na(`iCCA classification`) & `iCCA classification` != "",
                                        `iCCA classification`, "Unknown"))]
sankey_main <- sankey_dt[SubtypeClass == "Main",
                         .N, by = .(FinalSubtype, SiaSubclass, IGSubtype, DuctType)]
fwrite(sankey_main, file.path(data_dir, "sankey_main_subtype_sia_ig_duct_counts.csv"))

p_sankey <- ggplot(sankey_main,
                   aes(axis1 = FinalSubtype, axis2 = SiaSubclass, axis3 = IGSubtype, axis4 = DuctType, y = N)) +
  geom_alluvium(aes(fill = FinalSubtype), width = 0.18, alpha = 0.8, discern = TRUE) +
  geom_stratum(width = 0.18, fill = "grey92", color = "grey45", discern = TRUE) +
  geom_text(stat = "stratum", aes(label = after_stat(gsub("\\.\\d+$", "", stratum))),
            size = 3, discern = TRUE) +
  scale_x_discrete(limits = c("Subtype", "Sia", "IG", "Duct type"), expand = c(0.05, 0.05)) +
  scale_fill_manual(values = subtype_colors[main_subtypes]) +
  theme_pub(10) +
  theme(axis.text.x = element_text(angle = 0), legend.position = "none") +
  labs(title = "Subtype relationships with published and clinical annotations", x = NULL, y = "Samples")
ggsave(file.path(fig_dir, "sankey_main_subtype_sia_ig_duct.png"), p_sankey, width = 10, height = 6, dpi = 300)
p_sankey

### 11. Supportive multivariable survival models

In [ ]:
# Clinical and survival outputs are deliberately not imported here.
# This prevents an upstream transcriptomic notebook from depending on a downstream
# analysis and avoids preserving superseded Cox estimates as transcriptomic evidence.
clinical_owner <- file.path(project_root, "6-clinical-survival.ipynb")
message("Clinical/survival analysis is owned by: ", clinical_owner)


### 12. Unified source table for replacement panels

In [ ]:
# This section creates reusable boxplot-ready tables and replacement panels.
# Add requested variables from handwritten notes to `replacement_vars`.
replacement_sources <- list()

if (exists("metabolic_all")) {
  replacement_sources[["metabolic_all"]] <- metabolic_all$scores[, .(Sample, FinalSubtype, SubtypeClass, Panel = Signature, Value = Score)]
}
if (exists("stemness_all")) {
  replacement_sources[["stemness_all"]] <- stemness_all$scores[, .(Sample, FinalSubtype, SubtypeClass, Panel = Signature, Value = Score)]
}
if (file.exists(file.path(project_root, "governance/data/estimate_scores_with_final_subtype.csv"))) {
  est <- fread(file.path(project_root, "governance/data/estimate_scores_with_final_subtype.csv"))
  est_long <- melt(est,
                   id.vars = c("Sample", "FinalSubtype", "SubtypeClass"),
                   measure.vars = c("StromalScore", "ImmuneScore", "ESTIMATEScore", "TumorPurity"),
                   variable.name = "Panel", value.name = "Value")
  replacement_sources[["estimate"]] <- est_long
}

replacement_boxplot_dt <- rbindlist(replacement_sources, idcol = "Source", fill = TRUE)
fwrite(replacement_boxplot_dt, file.path(data_dir, "boxplot_replacement_source_table.csv"))

p_replace <- ggplot(replacement_boxplot_dt[SubtypeClass == "Main"],
                    aes(FinalSubtype, Value, fill = FinalSubtype)) +
  geom_boxplot(outlier.size = 0.4, alpha = 0.85) +
  facet_wrap(Source ~ Panel, scales = "free_y") +
  scale_fill_manual(values = subtype_colors[main_subtypes]) +
  theme_pub(8) +
  theme(legend.position = "none") +
  labs(title = "Boxplot replacement panels for main subtypes", x = NULL, y = "Value")
ggsave(file.path(fig_dir, "boxplot_replacement_panels_main_subtypes.png"), p_replace, width = 13, height = 9, dpi = 300)
p_replace

## Release checks and artifact manifest

In [ ]:
required <- c(
  file.path(step1_table_dir, "step1_main_only_study_adjusted_deg.csv"),
  file.path(step1_table_dir, "step1_ranked_gsea_main_only_study_adjusted.csv"),
  file.path(step1_table_dir, "step1_dorothea_tf_activity_main_only_study_adjusted.csv"),
  file.path(step1_table_dir, "step1_tme_concordance_scores_long.csv"),
  file.path(data_dir, "metabolic_main_subtypes_scores_long.csv"),
  file.path(data_dir, "stemness_main_subtypes_scores_long.csv"),
  file.path(data_dir, "tide_results_main_subtypes_annotated.csv")
)
stopifnot(all(file.exists(required)), all(file.info(required)$size > 0))

transcriptomic_gate <- data.table(
  GateItem = c(
    "Corrected DEG direction", "Ranked pathway analysis", "DoRothEA TF activity",
    "TME method concordance", "Metabolic and differentiation programs",
    "Intermediate-state separation", "TIDE interpretation", "Clinical and survival imports"
  ),
  Status = c("PASS", "PASS", "PASS", "PASS", "PASS", "PASS", "PASS_HYPOTHESIS_GENERATING", "DEFERRED_TO_OWN_GATES"),
  Blocking = FALSE
)
fwrite(transcriptomic_gate, file.path(report_dir, "transcriptomic_characterization_release_gate_status.csv"))
transcriptomic_gate

## Run summary

In [ ]:
summary_files <- list.files(data_dir, pattern = "\\.csv$|\\.xlsx$|\\.tsv$", full.names = TRUE)
figure_files <- list.files(fig_dir, pattern = "\\.png$|\\.pdf$", full.names = TRUE)

run_summary <- data.table(
  Item = c("Data files", "Figure files", "Main subtype samples", "Intermediate samples"),
  Value = c(length(summary_files), length(figure_files),
            nrow(main_meta), nrow(all_meta[SubtypeClass == "Intermediate"]))
)
fwrite(run_summary, file.path(report_dir, "transcriptomic_characterization_run_summary.csv"))

wb <- createWorkbook()
addWorksheet(wb, "Run Summary")
writeData(wb, "Run Summary", run_summary)
addWorksheet(wb, "31-gene signature")
writeData(wb, "31-gene signature", signature_table)
addWorksheet(wb, "Subtype counts")
writeData(wb, "Subtype counts", subtype_master[, .N, by = .(FinalSubtype, SubtypeClass)][order(SubtypeClass, FinalSubtype)])
addWorksheet(wb, "Metabolic KW")
writeData(wb, "Metabolic KW", metabolic_main$kruskal)
addWorksheet(wb, "Stemness KW")
writeData(wb, "Stemness KW", stemness_main$kruskal)
if (file.exists(file.path(data_dir, "multivariable_cox_main_subtypes_core.csv"))) {
  addWorksheet(wb, "Multivariable Cox")
  writeData(wb, "Multivariable Cox", fread(file.path(data_dir, "multivariable_cox_main_subtypes_core.csv")))
}
saveWorkbook(wb, file.path(data_dir, "transcriptomic_characterization_summary.xlsx"), overwrite = TRUE)

print(run_summary)
cat("\nTranscriptomic characterization notebook complete. Review files in:", out_dir, "\n")

In [ ]:
manifest_roots <- c(
  data_dir, fig_dir, report_dir,
  file.path(out_dir, "publication", "tables"),
  file.path(out_dir, "publication", "figures")
)
paths <- unlist(lapply(manifest_roots[dir.exists(manifest_roots)], list.files,
                       recursive = TRUE, full.names = TRUE), use.names = FALSE)
paths <- paths[file.exists(paths) & !dir.exists(paths) & !grepl("/__pycache__/", paths)]
manifest <- data.table(
  ArtifactType = fifelse(grepl("/figures/", paths), "figure", fifelse(grepl("/reports/", paths), "report", "source_table")),
  Path = sub(paste0("^", project_root, "/"), "", normalizePath(paths, winslash = "/", mustWork = TRUE)),
  Bytes = file.info(paths)$size
)
manifest[, `:=`(Exists = TRUE, NonEmpty = Bytes > 0)]
setorder(manifest, ArtifactType, Path)
fwrite(manifest, file.path(report_dir, "transcriptomic_characterization_output_manifest.csv"))
stopifnot(all(manifest$NonEmpty), !anyDuplicated(manifest$Path))
manifest[, .N, by = ArtifactType]

## Takeaways

- C1 metabolic/hepatic differentiation and C3 proliferative/epithelial programs are supported by corrected DEG, ranked pathways, TF activity, and derived program scores.
- C2 remains the stable identifier without an immune or ductal suffix; bulk ductal-lineage evidence requires malignant-cell and genomic reconciliation.
- Intermediate states are continuity evidence and are not collapsed into principal states.
- TIDE, mutation counts, drug-response projections, and imported clinical models remain supportive or hypothesis-generating rather than subtype-defining evidence.